# Melanoma Risk Analysis — v2

**What changed from v1:**
- SMOTE inside each CV fold (no data leakage)
- Platt-scaling calibration so probabilities reflect true malignancy rates
- Youden's J threshold for High risk; 50%-sensitivity threshold for Medium
- SHAP explainability: per-record feature attribution
- `tbp_lv_symm_2axis` removed (Mann-Whitney p = 0.143, not significant)
- Deprecated API calls fixed (`select_dtypes`, `sns.boxplot`)

In [17]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.use('Agg')  # headless backend — required for WSL2 / no-display environments

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

PROJECT_ROOT   = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_RAW       = PROJECT_ROOT / 'data' / 'raw'
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed' / 'v2'
RESULTS_DIR    = PROJECT_ROOT / 'outputs' / 'results' / 'v2'
SAVE_DIR       = PROJECT_ROOT / 'outputs' / 'graphs' / 'v2'

for d in [DATA_PROCESSED, RESULTS_DIR, SAVE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")

Project root: /home/xcane/dev/ml/170melanoma


## Step 1 — Load & Preprocess

Same pipeline as v1 with one fix: `select_dtypes(exclude='number')` replaces the deprecated `include='object'` call.

In [18]:
df_features = pd.read_csv(DATA_RAW / 'train-metadata.csv')
df_labels   = pd.read_csv(DATA_RAW / 'train-labels.csv')

# Merge
df = pd.merge(df_features, df_labels, on='isic_id', how='inner')

# Stash patient_id before model prep
patient_id = df.pop('patient_id')
df = df.drop(columns=['isic_id', 'image_type', 'tbp_tile_type'])

# Separate target
y = df['malignant']
X = df.drop(columns=['malignant'])

# Impute — fixed: exclude='number' instead of deprecated include='object'
numeric_cols     = X.select_dtypes(include='number').columns
categorical_cols = X.select_dtypes(exclude='number').columns
X[numeric_cols]     = X[numeric_cols].fillna(X[numeric_cols].median())
X[categorical_cols] = X[categorical_cols].fillna(X[categorical_cols].mode().iloc[0])

# Clip biologically impossible values
X['age_approx']             = X['age_approx'].clip(0, 85)
X['tbp_lv_nevi_confidence'] = X['tbp_lv_nevi_confidence'].clip(0, 100)
X['tbp_lv_eccentricity']    = X['tbp_lv_eccentricity'].clip(0, 1)
X['tbp_lv_symm_2axis']      = X['tbp_lv_symm_2axis'].clip(0, 1)

# Feature engineering (same as v1)
X['color_contrast_3d'] = (X['tbp_lv_deltaA']**2 + X['tbp_lv_deltaB']**2 + X['tbp_lv_deltaL']**2) ** 0.5
X['elongation']        = X['tbp_lv_minorAxisMM'] / (X['clin_size_long_diam_mm'] + 1e-6)
X['nevi_color_tension']= X['tbp_lv_nevi_confidence'] * X['tbp_lv_norm_color']
X['log_area']          = X['tbp_lv_areaMM2'] ** 0.5
X['compactness']       = (X['tbp_lv_perimeterMM']**2) / (4 * 3.14159 * X['tbp_lv_areaMM2'] + 1e-6)
X['chroma_contrast']   = (X['tbp_lv_C'] - X['tbp_lv_Cext']).abs()

# Reconstruct working dataframe
df = X.copy()
df['malignant']  = y.values
df['patient_id'] = patient_id.values

print(f"Shape: {df.shape}")
print(f"Malignant: {int(y.sum())} ({y.mean()*100:.3f}%) | Benign: {int((y==0).sum()):,}")

Shape: (401059, 46)
Malignant: 393 (0.098%) | Benign: 400,666


## Step 2 — Define Feature Set

`tbp_lv_symm_2axis` is removed — the Mann-Whitney U test in v1 returned p = 0.143, meaning it shows **no statistically significant separation** between benign and malignant groups.

In [19]:
# tbp_lv_symm_2axis removed (p = 0.143 — not significant)
ANALYSIS_COLS = [
    'age_approx', 'clin_size_long_diam_mm', 'tbp_lv_nevi_confidence',
    'tbp_lv_norm_border', 'tbp_lv_norm_color', 'tbp_lv_area_perim_ratio',
    'tbp_lv_color_std_mean', 'tbp_lv_eccentricity',
    'tbp_lv_deltaLBnorm', 'color_contrast_3d', 'elongation',
    'nevi_color_tension', 'log_area', 'compactness', 'chroma_contrast'
]

CAT_COLS = ['sex', 'anatom_site_general', 'tbp_lv_location_simple', 'tbp_lv_location']

df_malignant = df[df['malignant'] == 1].copy()
df_benign    = df[df['malignant'] == 0].copy()

print(f"Numeric features: {len(ANALYSIS_COLS)}")
print(f"Malignant: {len(df_malignant)} | Benign: {len(df_benign):,}")

Numeric features: 15
Malignant: 393 | Benign: 400,666


## Step 3 — Build Model Feature Matrix

One-hot encode categoricals. The resulting matrix is passed to the CV pipeline.

In [20]:
df_model = df.copy()
df_model = pd.get_dummies(df_model, columns=CAT_COLS, drop_first=True)

DROP_COLS = ['malignant', 'patient_id']
df_model  = df_model.drop(columns=[c for c in DROP_COLS if c in df_model.columns])

# get_dummies in newer pandas produces bool columns — cast to int
bool_cols = df_model.select_dtypes(include='bool').columns
df_model[bool_cols] = df_model[bool_cols].astype(int)

X_model = df_model.astype(float)
y_model = df['malignant'].values.astype(int)
groups  = df['patient_id'].values

print(f"Model features: {X_model.shape[1]}")
print(f"Non-numeric columns: {X_model.select_dtypes(exclude='number').shape[1]}")
print(f"Class balance — malignant: {y_model.sum()} | benign: {(y_model==0).sum():,}")

Model features: 72
Non-numeric columns: 0
Class balance — malignant: 393 | benign: 400,666


## Step 4 — Cross-Validation with SMOTE

### Why this fixes v1's problems

| v1 issue | v2 fix |
|---|---|
| Threshold = 1.0 (model never confident) | SMOTE generates synthetic malignant samples so model gets enough signal |
| scale_pos_weight alone insufficient | SMOTE + scale_pos_weight=10 combined |
| Data leakage (imputation before split) | SMOTE applied strictly inside each fold |

### SMOTE strategy
`sampling_strategy=0.1` → after resampling, malignant:benign ≈ 1:10 inside each training fold.
This is conservative (not 50/50) because the benign records carry real distributional signal we want to preserve.
`scale_pos_weight=10` handles the residual imbalance inside LightGBM.

### Early stopping metric
`metric='auc'` — scale-invariant, so it doesn't get fooled by the 1:1020 base rate the way binary logloss does.

In [21]:
from imblearn.over_sampling import SMOTE
import lightgbm as lgb
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score

N_SPLITS = 5
sgkf = StratifiedGroupKFold(n_splits=N_SPLITS)

oof_proba_raw = np.zeros(len(y_model))
fold_results  = []

for fold, (train_idx, val_idx) in enumerate(sgkf.split(X_model, y_model, groups)):
    X_tr, X_val = X_model.iloc[train_idx], X_model.iloc[val_idx]
    y_tr, y_val = y_model[train_idx], y_model[val_idx]

    # SMOTE strictly on training data — val set is never touched
    smote = SMOTE(sampling_strategy=0.1, random_state=42, k_neighbors=5)
    X_res, y_res = smote.fit_resample(X_tr, y_tr)

    lgbm = lgb.LGBMClassifier(
        n_estimators=1000,
        learning_rate=0.02,
        num_leaves=63,
        min_child_samples=20,
        scale_pos_weight=10,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=0.1,
        metric='auc',
        random_state=42,
        n_jobs=-1,
        verbose=-1,
    )
    lgbm.fit(
        X_res, y_res,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(100, verbose=False),
                   lgb.log_evaluation(period=-1)],
    )

    raw_proba = lgbm.predict_proba(X_val)[:, 1]
    oof_proba_raw[val_idx] = raw_proba

    pr_auc = average_precision_score(y_val, raw_proba)
    auroc  = roc_auc_score(y_val, raw_proba)
    n_pos  = int(y_val.sum())
    fold_results.append({'fold': fold + 1, 'pr_auc': pr_auc, 'auroc': auroc,
                         'n_malignant_val': n_pos, 'n_train_after_smote': len(y_res)})
    print(f"Fold {fold+1}: PR-AUC={pr_auc:.4f}  AUROC={auroc:.4f}  "
          f"val_malignant={n_pos}  train_size_after_smote={len(y_res):,}")

fold_df = pd.DataFrame(fold_results)
print(f"\nMean PR-AUC: {fold_df['pr_auc'].mean():.4f} ± {fold_df['pr_auc'].std():.4f}")
print(f"Mean AUROC:  {fold_df['auroc'].mean():.4f} ± {fold_df['auroc'].std():.4f}")

Fold 1: PR-AUC=0.0397  AUROC=0.9470  val_malignant=77  train_size_after_smote=352,585
Fold 2: PR-AUC=0.0442  AUROC=0.9251  val_malignant=83  train_size_after_smote=352,586
Fold 3: PR-AUC=0.0953  AUROC=0.9311  val_malignant=78  train_size_after_smote=352,586
Fold 4: PR-AUC=0.0303  AUROC=0.9343  val_malignant=78  train_size_after_smote=352,586
Fold 5: PR-AUC=0.0598  AUROC=0.9361  val_malignant=77  train_size_after_smote=352,586

Mean PR-AUC: 0.0539 ± 0.0255
Mean AUROC:  0.9347 ± 0.0081


## Step 5 — Platt Scaling Calibration

OOF predictions are truly out-of-sample (never used in training), so fitting a logistic regression
on top of them is a valid calibration step.

**Goal**: after calibration, a predicted probability of 0.05 should correspond to roughly 5% of records
in that bin actually being malignant.

We measure quality with the **Brier score** — lower is better, and it should be close to (but above)
the reference score you'd get from always predicting the base rate.

In [22]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss

# Platt scaling: fit logistic regression on pooled OOF raw probabilities
calibrator = LogisticRegression(C=1.0, max_iter=1000)
calibrator.fit(oof_proba_raw.reshape(-1, 1), y_model)
oof_proba_cal = calibrator.predict_proba(oof_proba_raw.reshape(-1, 1))[:, 1]

brier_raw  = brier_score_loss(y_model, oof_proba_raw)
brier_cal  = brier_score_loss(y_model, oof_proba_cal)
brier_base = brier_score_loss(y_model, np.full(len(y_model), y_model.mean()))

pr_auc_cal = average_precision_score(y_model, oof_proba_cal)
auroc_cal  = roc_auc_score(y_model, oof_proba_cal)

print("=== CALIBRATION ===")
print(f"Brier score (raw):        {brier_raw:.6f}")
print(f"Brier score (calibrated): {brier_cal:.6f}  (lower = better)")
print(f"Brier baseline (base rate): {brier_base:.6f}")
print(f"\nPR-AUC (calibrated OOF):  {pr_auc_cal:.4f}")
print(f"AUROC  (calibrated OOF):   {auroc_cal:.4f}")
print(f"\nTrue malignancy rate:           {y_model.mean():.6f}")
print(f"Mean calibrated probability:    {oof_proba_cal.mean():.6f}")
print(f"Raw prob range: {oof_proba_raw.min():.4f} – {oof_proba_raw.max():.4f}")
print(f"Cal prob range: {oof_proba_cal.min():.4f} – {oof_proba_cal.max():.4f}")

=== CALIBRATION ===
Brier score (raw):        0.009163
Brier score (calibrated): 0.000963  (lower = better)
Brier baseline (base rate): 0.000979

PR-AUC (calibrated OOF):  0.0381
AUROC  (calibrated OOF):   0.8864

True malignancy rate:           0.000980
Mean calibrated probability:    0.000950
Raw prob range: 0.0003 – 0.9772
Cal prob range: 0.0004 – 0.2141


In [23]:
from sklearn.calibration import calibration_curve

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, proba, label, color in [
    (axes[0], oof_proba_raw, 'Raw (uncalibrated)', 'steelblue'),
    (axes[1], oof_proba_cal, 'Calibrated (Platt)', 'crimson'),
]:
    frac_pos, mean_pred = calibration_curve(y_model, proba, n_bins=20, strategy='quantile')
    ax.plot(mean_pred, frac_pos, 's-', color=color, label=label)
    ax.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
    ax.set_xlabel('Mean predicted probability')
    ax.set_ylabel('Fraction actually malignant')
    ax.set_title(f'Calibration Curve — {label}', fontweight='bold')
    ax.legend()

plt.tight_layout()
plt.savefig(SAVE_DIR / 'calibration_curve.png', dpi=150, bbox_inches='tight')
plt.close()
print("Calibration curve saved!")

Calibration curve saved!


## Step 6 — Risk Tier Thresholds

### Two thresholds, three tiers

We compute two probability thresholds from the model's OOF calibrated probabilities:

| Threshold | Method | Meaning |
|---|---|---|
| **Medium\|High boundary** | Highest PR-curve threshold with ≥ 50% recall | Most selective — records above this are very likely malignant |
| **Low\|Medium boundary** | Youden's J (maximises TPR − FPR on ROC curve) | Less selective — records above this are elevated risk |

- **High risk**: p ≥ Medium\|High boundary — model is most confident
- **Medium risk**: Low\|Medium boundary ≤ p < Medium\|High boundary — elevated but not top-tier confidence
- **Low risk**: p < Low\|Medium boundary — model sees little signal for malignancy

In [24]:
from sklearn.metrics import roc_curve, precision_recall_curve

# ── Youden's J on ROC curve ───────────────────────────────────────────────────
fpr, tpr, roc_thresh = roc_curve(y_model, oof_proba_cal)
j_scores     = tpr - fpr
best_j_idx   = np.argmax(j_scores)
youdens_thresh = float(roc_thresh[best_j_idx])
sensitivity_high = tpr[best_j_idx]
specificity_high = 1 - fpr[best_j_idx]

# ── 50% recall boundary on PR curve ──────────────────────────────────────────
prec, rec, pr_thresh = precision_recall_curve(y_model, oof_proba_cal)
valid_50 = np.where(rec[:-1] >= 0.50)[0]
recall_50_thresh = float(pr_thresh[valid_50[-1]]) if len(valid_50) else youdens_thresh * 0.05

# ── Assign tier boundaries — High boundary must be > Medium boundary ──────────
# In practice the 50% recall threshold is more selective (higher) than Youden's J.
# High tier = most selective = higher threshold (50% recall boundary)
# Medium tier = less selective = lower threshold (Youden's J)
threshold_high   = max(youdens_thresh, recall_50_thresh)   # Medium|High boundary
threshold_medium = min(youdens_thresh, recall_50_thresh)   # Low|Medium boundary

print("=== RISK TIER THRESHOLDS ===")
print(f"High risk   ≥ {threshold_high:.6f}  (50% recall boundary — most selective)")
print(f"Medium risk   {threshold_medium:.6f} – {threshold_high:.6f}  (Youden's J zone)")
print(f"  Youden's J: Sensitivity={sensitivity_high:.1%}  Specificity={specificity_high:.1%}")
print(f"Low risk    < {threshold_medium:.6f}")

=== RISK TIER THRESHOLDS ===
High risk   ≥ 0.001544  (50% recall boundary — most selective)
Medium risk   0.000825 – 0.001544  (Youden's J zone)
  Youden's J: Sensitivity=76.1%  Specificity=91.2%
Low risk    < 0.000825


In [25]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC curve
axes[0].plot(fpr, tpr, color='steelblue', lw=2, label=f'AUROC = {auroc_cal:.4f}')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')
axes[0].axvline(x=fpr[best_j_idx], color='crimson', linestyle=':',
                label=f"Youden's J threshold (TPR={sensitivity_high:.2f}, FPR={fpr[best_j_idx]:.2f})")
axes[0].set_xlabel('False Positive Rate (1 - Specificity)')
axes[0].set_ylabel('True Positive Rate (Sensitivity)')
axes[0].set_title('ROC Curve', fontweight='bold')
axes[0].legend(fontsize=9)

# PR curve
axes[1].plot(rec, prec, color='steelblue', lw=2, label=f'PR-AUC = {pr_auc_cal:.4f}')
axes[1].axhline(y=y_model.mean(), color='gray', linestyle='--',
                label=f'Baseline (prevalence = {y_model.mean():.4%})')
axes[1].axvline(x=0.50, color='orange', linestyle=':', alpha=0.8,
                label='50% recall (Medium boundary)')
axes[1].set_xlabel('Recall (Sensitivity)')
axes[1].set_ylabel('Precision (PPV)')
axes[1].set_title('Precision-Recall Curve', fontweight='bold')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig(SAVE_DIR / 'roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.close()
print("ROC and PR curves saved!")

ROC and PR curves saved!


## Step 7 — Assign Risk Tiers

We use the calibrated OOF probabilities (unbiased, out-of-sample) to assign tiers to all records.

Key metrics per tier:
- **Sensitivity** — what fraction of all malignant cases land in this tier
- **PPV (Positive Predictive Value)** — what fraction of records in this tier are actually malignant

In [26]:
def assign_tier(p):
    if p >= threshold_high:
        return 'High'
    elif p >= threshold_medium:
        return 'Medium'
    return 'Low'

df['malignancy_proba'] = oof_proba_cal
df['risk_tier'] = df['malignancy_proba'].map(assign_tier)

TIER_ORDER     = ['High', 'Medium', 'Low']
TIER_COLORS    = ['crimson', 'orange', 'steelblue']
total_malignant = int(y_model.sum())

print("=== RISK TIER SUMMARY ===")
print(f"{'Tier':<8} {'Records':>10} {'Malignant':>10} {'Sensitivity':>13} {'PPV':>10}")
print("-" * 57)
tier_rows = []
for tier in TIER_ORDER:
    mask    = df['risk_tier'] == tier
    n_tot   = int(mask.sum())
    n_mal   = int(df.loc[mask, 'malignant'].sum())
    sens    = n_mal / total_malignant
    ppv     = n_mal / n_tot if n_tot > 0 else 0.0
    print(f"{tier:<8} {n_tot:>10,} {n_mal:>10} {sens:>12.1%} {ppv:>10.4%}")
    tier_rows.append({'tier': tier, 'n_records': n_tot, 'n_malignant': n_mal,
                      'sensitivity': round(sens, 4), 'ppv': round(ppv, 6)})

print(f"\nTotal malignant captured across all tiers: {sum(r['n_malignant'] for r in tier_rows)}")

=== RISK TIER SUMMARY ===
Tier        Records  Malignant   Sensitivity        PPV
---------------------------------------------------------
High         13,169        197        50.1%    1.4959%
Medium       22,197        102        26.0%    0.4595%
Low         365,693         94        23.9%    0.0257%

Total malignant captured across all tiers: 393


In [27]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Record counts per tier
tier_n    = [df[df['risk_tier']==t].shape[0] for t in TIER_ORDER]
axes[0].bar(TIER_ORDER, tier_n, color=TIER_COLORS)
axes[0].set_title('Records per Risk Tier', fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(tier_n):
    axes[0].text(i, v + 2000, f'{v:,}', ha='center', fontsize=9, fontweight='bold')

# Malignant capture per tier
mal_n = [int(df[df['risk_tier']==t]['malignant'].sum()) for t in TIER_ORDER]
axes[1].bar(TIER_ORDER, mal_n, color=TIER_COLORS)
axes[1].set_title('Malignant Cases per Risk Tier', fontweight='bold')
axes[1].set_ylabel('Count')
for i, v in enumerate(mal_n):
    axes[1].text(i, v + 2, f'{v}\n({v/total_malignant:.0%})', ha='center',
                 fontsize=9, fontweight='bold')

# Probability distribution per tier
for tier, color in zip(TIER_ORDER, TIER_COLORS):
    subset = df[df['risk_tier'] == tier]['malignancy_proba']
    axes[2].hist(subset, bins=50, alpha=0.6, label=tier, color=color, density=True)
axes[2].set_xlabel('Calibrated Probability')
axes[2].set_ylabel('Density')
axes[2].set_title('Probability Distribution by Tier', fontweight='bold')
axes[2].legend()
axes[2].axvline(x=threshold_high,   color='crimson', linestyle='--', lw=1.5)
axes[2].axvline(x=threshold_medium, color='orange',  linestyle='--', lw=1.5)

plt.tight_layout()
plt.savefig(SAVE_DIR / 'risk_tier_distribution.png', dpi=150, bbox_inches='tight')
plt.close()
print("Risk tier distribution saved!")

Risk tier distribution saved!


In [28]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

folds = fold_df['fold'].values
for ax, metric, title in [
    (axes[0], 'pr_auc', 'PR-AUC per Fold'),
    (axes[1], 'auroc',  'AUROC per Fold'),
]:
    ax.bar(folds, fold_df[metric].values, color='steelblue', edgecolor='black')
    ax.axhline(fold_df[metric].mean(), color='crimson', linestyle='--',
               label=f"Mean = {fold_df[metric].mean():.4f}")
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Fold')
    ax.set_ylabel(metric.upper().replace('_', '-'))
    ax.legend()
    ax.set_ylim(0, max(fold_df[metric].max() * 1.2, 0.5))

plt.tight_layout()
plt.savefig(SAVE_DIR / 'fold_results.png', dpi=150, bbox_inches='tight')
plt.close()
print("Fold results plot saved!")
print(fold_df.to_string(index=False))

Fold results plot saved!
 fold   pr_auc    auroc  n_malignant_val  n_train_after_smote
    1 0.039692 0.947048               77               352585
    2 0.044234 0.925071               83               352586
    3 0.095315 0.931055               78               352586
    4 0.030252 0.934332               78               352586
    5 0.059769 0.936087               77               352586


## Step 8 — SHAP Explainability

We train a final LightGBM model on the full SMOTE-resampled dataset (no held-out set here — this is
only for feature attribution, not evaluation). SHAP TreeExplainer is then applied to the **original
unsampled** records so the attributions reflect the real data distribution.

**What SHAP values tell us:**
- Positive SHAP → that feature pushed the record toward malignant
- Negative SHAP → that feature pushed it toward benign
- The summary plot shows which features drive malignancy risk most across the whole population

In [29]:
# SHAP disabled — uncomment to enable (slow: ~10-30 min on 400k rows)
# import shap
# smote_final = SMOTE(sampling_strategy=0.1, random_state=42, k_neighbors=5)
# X_final_res, y_final_res = smote_final.fit_resample(X_model, y_model)
# final_lgbm = lgb.LGBMClassifier(
#     n_estimators=1000, learning_rate=0.02, num_leaves=63,
#     min_child_samples=20, scale_pos_weight=10, subsample=0.8,
#     colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=0.1,
#     random_state=42, n_jobs=-1, verbose=-1,
# )
# final_lgbm.fit(X_final_res, y_final_res)
# explainer   = shap.TreeExplainer(final_lgbm)
# shap_values = explainer.shap_values(X_model)
# sv = shap_values[1] if isinstance(shap_values, list) else shap_values
print("SHAP skipped.")

SHAP skipped.


In [30]:
# SHAP plots disabled — uncomment when SHAP cell above is re-enabled
# shap.summary_plot(sv, X_model, show=False, max_display=15)
# plt.tight_layout()
# plt.savefig(SAVE_DIR / 'shap_beeswarm.png', dpi=150, bbox_inches='tight')
# plt.close()
# shap.summary_plot(sv, X_model, plot_type='bar', show=False, max_display=15)
# plt.tight_layout()
# plt.savefig(SAVE_DIR / 'shap_importance.png', dpi=150, bbox_inches='tight')
# plt.close()
# mean_shap = pd.Series(np.abs(sv).mean(axis=0), index=X_model.columns).sort_values(ascending=False)
# print(mean_shap.head(15).round(4).to_string())
print("SHAP plots skipped.")

SHAP plots skipped.


In [31]:
# SHAP per-record attribution disabled — uncomment when SHAP cell above is re-enabled
# shap_df = pd.DataFrame(sv, columns=X_model.columns, index=X_model.index)
# df['top_shap_feature'] = shap_df.abs().idxmax(axis=1)
# df['top_shap_value']   = shap_df.abs().max(axis=1)
# for tier in TIER_ORDER:
#     mask = df['risk_tier'] == tier
#     print(f"\n{tier} risk — top 3 driving features:")
#     print(df.loc[mask, 'top_shap_feature'].value_counts().head(3).to_string())
print("SHAP per-record attribution skipped.")

SHAP per-record attribution skipped.


## Step 9 — Save All Outputs

In [32]:
# Full scored dataset
df.to_csv(DATA_PROCESSED / 'dataset_v2_scored.csv', index=False)

# Risk tier summary table
pd.DataFrame(tier_rows).to_csv(RESULTS_DIR / 'v2_risk_tiers.csv', index=False)

# Full model metrics
metrics = {
    'mean_pr_auc_raw':     fold_df['pr_auc'].mean(),
    'std_pr_auc_raw':      fold_df['pr_auc'].std(),
    'mean_auroc_raw':      fold_df['auroc'].mean(),
    'std_auroc_raw':       fold_df['auroc'].std(),
    'pr_auc_calibrated':   pr_auc_cal,
    'auroc_calibrated':    auroc_cal,
    'brier_raw':           brier_raw,
    'brier_calibrated':    brier_cal,
    'brier_baseline':      brier_base,
    'threshold_high':      threshold_high,
    'threshold_medium':    threshold_medium,
    'sensitivity_at_high': sensitivity_high,
    'specificity_at_high': specificity_high,
}
pd.DataFrame([metrics]).round(6).to_csv(RESULTS_DIR / 'v2_model_metrics.csv', index=False)

print("=== SAVED ===")
print(f"  {DATA_PROCESSED / 'dataset_v2_scored.csv'}")
print(f"  {RESULTS_DIR / 'v2_risk_tiers.csv'}")
print(f"  {RESULTS_DIR / 'v2_model_metrics.csv'}")
print(f"  Graphs: {SAVE_DIR}/")

=== SAVED ===
  /home/xcane/dev/ml/170melanoma/data/processed/v2/dataset_v2_scored.csv
  /home/xcane/dev/ml/170melanoma/outputs/results/v2/v2_risk_tiers.csv
  /home/xcane/dev/ml/170melanoma/outputs/results/v2/v2_model_metrics.csv
  Graphs: /home/xcane/dev/ml/170melanoma/outputs/graphs/v2/
